In [2]:
!gdown --id 1Xw0xNiNZYgARcWja7LRrYR__IpiF5tCn

/usr/local/lib/python3.10/dist-packages/gdown/cli.py:138: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1Xw0xNiNZYgARcWja7LRrYR__IpiF5tCn
To: /content/EEG-CNN-Epilepsy.zip
100% 1.85M/1.85M [00:00<00:00, 180MB/s]


In [3]:
!unzip -q EEG-CNN-Epilepsy.zip

In [ ]:
%cd extracted_EEG-CNN-Epilepsy

In [5]:
%run EEG_dataset_module.py

In [6]:
import torch
from torch.utils.data import DataLoader
from EEG_dataset_module import CustomDataset,ToTensor,Filter
import torch.nn as nn
from torchvision.transforms import Compose
# import matplotlib.pyplot as plt
from torch import optim

In [7]:
import glob
pathname= '/content/train'

file_names= glob.glob(pathname+ '/*/*.txt')

In [8]:
file_names[0]

'/content/train/Z/Z039.txt'

In [9]:
import torch
from torch.utils.data import Dataset
import glob
import pandas as pd
from scipy import signal
import numpy as np

class CustomDataset(Dataset):
    def __init__(self,pathname,transform=None,target_transform=None):
        self.file_names= glob.glob(pathname+ '/*/*.txt')
        self.class_map= {'S':0,'Z':1,'F':2}
        self.transform= transform
        self.target_transform= target_transform

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self,indx):
        filename= self.file_names[indx]
        df= pd.read_csv(filename,header=None)
        chr= filename.split('/')[-1][0]

        sample= df.iloc[:,0].to_numpy()
        label= self.class_map[chr]
        if self.transform:
            sample= self.transform(sample)
        if self.target_transform:
            label= self.target_transform(label)
        return sample,label

class ToTensor:
    def __call__(self, sample):
        # sample = torch.from_numpy(sample).type(torch.float32)
        sample = torch.tensor(sample.copy(),dtype=torch.float32)
        return sample

class Filter:
    def __init__(self,Fs,order,fl,fh,type):
        self.fs=Fs
        self.wn= np.array([fl,fh])/(Fs/2)
        self.b,self.a= signal.butter(order, self.wn,btype=type)

    def __call__(self, sample):
        sample = signal.filtfilt(self.b,self.a,sample)
        return sample

class Normalize:
    def __init__(self,mu,std):
        self.mean= mu
        self.std= std

    def __call__(self,sample):
        sample= (sample-self.mean)/self.std
        return sample

In [10]:
pathname='/content/train'
transfrom= Compose((Filter(Fs=173.61,fl=8,fh=15,order=3,type='bandpass'),ToTensor()))
ds_train= CustomDataset(pathname,transfrom)

pathname='/content/test'
transfrom= Compose((Filter(Fs=173.61,fl=8,fh=15,order=3,type='bandpass'),ToTensor()))
ds_test= CustomDataset(pathname,transfrom)

In [11]:
train_loader= DataLoader(ds_train,batch_size=32,
                         drop_last=False,shuffle=True,num_workers=0)

test_loader= DataLoader(ds_test,batch_size=32,
                         drop_last=False,shuffle=False,num_workers=0)

In [12]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet,self).__init__()
        self.conv1= nn.Conv1d(in_channels=1, out_channels=15, kernel_size=5,padding=2)
        self.pool1= nn.MaxPool1d(4)
        self.conv2= nn.Conv1d(in_channels=15, out_channels=20, kernel_size=5,padding=2)
        self.pool2= nn.MaxPool1d(4)

        self.fc1= nn.Linear(20*256, 200)
        self.fc2= nn.Linear(200, 84)
        self.fc3= nn.Linear(84, 50)
        self.fc4= nn.Linear(50, 3)
        self.tanh= nn.Tanh()
    def forward(self,x):
        out=  self.tanh(self.conv1(x))
        out=  self.pool1(out)
        out=  self.tanh(self.conv2(out))
        out=  self.pool2(out)

        features= out.flatten(start_dim=1)
        out= self.tanh(self.fc1(features))
        out= self.tanh(self.fc2(out))
        out= self.tanh(self.fc3(out))
        out= (self.fc4(out))
        return out,features

In [13]:
 #%% define hyperparameters
lr=0.001 # the value of learning rate is determined by the user,
model= ConvNet()
#%% import MNIST dataset
optimizer= optim.SGD(params=model.parameters(),lr=0.01)
# optimizer= optim.SGD(params=model.parameters(),lr=lr,momentum=0.2)
# optimizer= optim.Adam(params=model.parameters(),lr=lr,betas=(0.9,0.999),eps= 1e-8)
# optimizer= optim.RMSprop(params=model.parameters(),lr=lr)
criteria= nn.CrossEntropyLoss()

In [14]:
#%% train Neural Network
mse=[]
epoch=80 # number of training iteration
for iter in range(1,epoch):
     er=[]
     for i,(xbatch,ybatch) in enumerate(train_loader):
          # farward pass
          xbatch= xbatch.unsqueeze(1)
          # xbatch= xbatch.unsqueeze(1)
          ypred,_= model(xbatch)
          # backward pass
          loss=  criteria(ypred,ybatch)
          loss.backward() # triger gradient calculation
          optimizer.step()# update parameters(synaptic wieghts)
          optimizer.zero_grad() # clear gradients

          er.append(loss.detach())

     loss= torch.mean(torch.tensor(er))
     mse.append(loss)
     # tt= torch.mean(torch.tensor(elapsed))
     print(f'MSE({iter-1}): {mse[iter-1]:.5f}')

mse=torch.tensor(mse)
# print(f'Averaged elapased time: {torch.mean(torch.tensor(elapsed)):.5f}')


MSE(0): 1.10195
MSE(1): 1.09925
MSE(2): 1.09728
MSE(3): 1.09577
MSE(4): 1.09457
MSE(5): 1.09207
MSE(6): 1.08963
MSE(7): 1.08662
MSE(8): 1.08912
MSE(9): 1.08007
MSE(10): 1.08209
MSE(11): 1.07945
MSE(12): 1.07332
MSE(13): 1.07088
MSE(14): 1.06398
MSE(15): 1.06345
MSE(16): 1.05639
MSE(17): 1.05181
MSE(18): 1.04543
MSE(19): 1.03621
MSE(20): 1.02878
MSE(21): 1.02191
MSE(22): 1.01037
MSE(23): 0.99696
MSE(24): 0.98628
MSE(25): 0.98042
MSE(26): 0.95637
MSE(27): 0.93611
MSE(28): 0.91497
MSE(29): 0.88400
MSE(30): 0.86372
MSE(31): 0.82961
MSE(32): 0.82045
MSE(33): 0.76782
MSE(34): 0.80682
MSE(35): 0.72888
MSE(36): 0.71717
MSE(37): 0.68232
MSE(38): 0.67045
MSE(39): 0.65743
MSE(40): 0.69559
MSE(41): 0.59898
MSE(42): 0.59731
MSE(43): 0.59281
MSE(44): 0.57561
MSE(45): 0.54374
MSE(46): 0.51349
MSE(47): 0.50283
MSE(48): 0.52451
MSE(49): 0.51998
MSE(50): 0.50432
MSE(51): 0.46160
MSE(52): 0.41908
MSE(53): 0.39529
MSE(54): 0.38389
MSE(55): 0.35590
MSE(56): 0.35011
MSE(57): 0.31823
MSE(58): 0.29740
MSE(59)

In [15]:
#%% test trained Neural Network
with torch.no_grad():
    n_sample=0
    n_correct=0
    for i,(samples,labels) in enumerate(test_loader):
        # forward pass
        samples= samples.unsqueeze(1)
        y_hat,_= model(samples)
        y_hat= torch.argmax(y_hat,dim=1)
        n_correct+=  torch.sum(y_hat==labels)
        n_sample+=labels.size(0)

accuracy= n_correct/n_sample *100
print('Accuracy:',accuracy)

Accuracy: tensor(97.7778)
